1) 데이터 로드 (HuggingFace OK)

In [38]:
from datasets import load_dataset
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import re

In [39]:
raw_datasets = load_dataset("imdb")

2) tokenizer (RNN용 → 반드시 word-level)

In [40]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text.split()

3) vocab 생성 (중요)

In [41]:
counter = Counter()

for text in raw_datasets["train"]["text"]:
    counter.update(tokenize(text))

vocab = {"<PAD>": 0, "<UNK>": 1}

for word, _ in counter.most_common(30000):
    vocab[word] = len(vocab)

vocab_size = len(vocab)

4) encode 함수 (padding 포함)

In [42]:
MAX_LEN = 500

def encode(text):
    tokens = tokenize(text)

    ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]

    if len(ids) < MAX_LEN:
        ids += [0] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]

    return ids

5) Dataset 만들기

In [44]:
class IMDBDataset(Dataset):
    def __init__(self, dataset):
        self.texts = dataset["text"]
        self.labels = dataset["label"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

6) split + loader

In [45]:
split = raw_datasets["train"].train_test_split(test_size=0.2, seed=42)

train_ds = IMDBDataset(split["train"])
valid_ds = IMDBDataset(split["test"])
test_ds = IMDBDataset(raw_datasets["test"])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)

7) 🔥 핵심 모델 (RNN / LSTM / GRU 공통)

In [46]:
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim,
                 model_type="LSTM", bidirectional=True, dropout=0.3):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        self.model_type = model_type
        self.bidirectional = bidirectional
        self.num_dir = 2 if bidirectional else 1

        if model_type == "RNN":
            self.rnn = nn.RNN(emb_dim, hidden_dim, batch_first=True,
                              bidirectional=bidirectional)

        elif model_type == "LSTM":
            self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True,
                               bidirectional=bidirectional)

        elif model_type == "GRU":
            self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True,
                              bidirectional=bidirectional)

        self.fc = nn.Linear(hidden_dim * self.num_dir, 2)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        emb = self.embedding(x)

        if self.model_type == "LSTM":
            output, (hidden, cell) = self.rnn(emb)
        else:
            output, hidden = self.rnn(emb)

        # 마지막 hidden 사용 (bidirectional 고려)
        hidden = hidden[-self.num_dir:]
        hidden = torch.cat([h for h in hidden], dim=1)

        return self.fc(self.dropout(hidden))

8) 모델 생성 (비교 핵심 부분)

In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def build_model(model_type):
    return SentimentModel(
        vocab_size=vocab_size,
        emb_dim=128,
        hidden_dim=256,
        model_type=model_type,
        bidirectional=True
    ).to(device)

9) training loop (공통)

In [48]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)

        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

10) evaluation

In [49]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            out = model(x)
            pred = out.argmax(dim=1)

            correct += (pred == y).sum().item()
            total += y.size(0)

    return correct / total

11) 🔥 비교 실험 실행 코드

In [50]:
for model_type in ["RNN", "GRU", "LSTM"]:

    print(f"\n=== {model_type} ===")

    model = build_model(model_type)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(3):
        loss = train(model, train_loader, optimizer, criterion)
        acc = evaluate(model, valid_loader)

        print(f"Epoch {epoch+1} | Loss {loss:.4f} | Acc {acc:.4f}")


=== RNN ===
Epoch 1 | Loss 0.7080 | Acc 0.5058
Epoch 2 | Loss 0.7030 | Acc 0.5020
Epoch 3 | Loss 0.6995 | Acc 0.6184

=== GRU ===
Epoch 1 | Loss 0.6210 | Acc 0.7672
Epoch 2 | Loss 0.3727 | Acc 0.8684
Epoch 3 | Loss 0.2145 | Acc 0.8788

=== LSTM ===
Epoch 1 | Loss 0.6463 | Acc 0.7102
Epoch 2 | Loss 0.5737 | Acc 0.7050
Epoch 3 | Loss 0.3396 | Acc 0.8634
